### Step 1: 	Hello, Data! Load raw CSV, display first 3 rows

1. Chatgpt was used to add new column named "ShippingAddressID" is added to the dataset and the rows are populated with a 5-digit unique numbers to be served as foreign key to another data set containing the shipping details

2. Another column named "CouponCode" is added programatically, and the rows are populated with another 5 digit random keys.

In [423]:
from importlib import reload
import random
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

import modules.FileManager
reload(modules.FileManager)

import modules.DBManager
reload(modules.DBManager)

# Path to the CSV file containing 
sales_records_path = "data/1000_sales_records_with_shipping_address_id.csv"
address_path = "data/shipping_addresses.csv"
num_of_rows_needed = 500 # As per assignment requirement says
#Loads the CSV file
fileManager = modules.FileManager.FileManager(sales_records_path, num_of_rows_needed)
sales_data = fileManager.get_data()

address_data = pd.read_csv(address_path)

# Filter addresses that match the 500 rows
filtered_addresses = address_data[
    address_data["shipping_address_id"].isin(sales_data["Shipping Address ID"])
]

# Get the number of rows in the DataFrame
num_rows = len(filtered_addresses)

db_manager = modules.DBManager.DBManager(filtered_addresses)
# db_manager._drop_and_create_table()
# db_manager._insert_into_table()
addresses_from_database = db_manager._fect_all()

# Merge sales_data with address_data on 'Shipping Address ID'
sales_shipping_data = pd.merge(sales_data, addresses_from_database, left_on='Shipping Address ID', right_on='shipping_address_id', how='inner')

# Print the first 3 rows of the data
sales_shipping_data.head(3)


✅ Successfully loaded 500 rows from data/1000_sales_records_with_shipping_address_id.csv


c:\Users\MOSTAFA\Desktop\ML Programming\Lab 2\modules\DBManager.py:74: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(sql_query, conn)


✅ Successfully loaded 500 rows from Shipping_Addresses table.


,Region,Country,Item Type,Sales Channel,Order Priority,Order Date,Order ID,Ship Date,Units Sold,Unit Price,...,Total Cost,Total Profit,Shipping Address ID,shipping_address_id,country,street,city,state/province,postal_code,phone_number
0,Middle East and North Africa,Libya,Cosmetics,Offline,M,10/18/2014,686800706,10/31/2014,8446,437.20,...,2224085.18,1468506.02,85831,85831,Libya,1069 Main St,Sydney,ENG,33769.0,Phone Number
1,North America,Canada,Vegetables,Online,M,11/7/2011,185941302,12/8/2011,3018,154.06,...,274426.74,190526.34,72252,72252,Canada,8872 Main St,Toronto,NY,98001.0,Phone Number
2,Middle East and North Africa,Libya,Baby Food,Offline,C,10/31/2016,246222341,12/9/2016,1517,255.28,...,241840.14,145419.62,92369,92369,Libya,2924 Main St,Tokyo,BE,97404.0,Phone Number


### Step 2: Pick the Right Container. Justify dict vs namedtuple vs sets(1–2 sentences)

1. Dictionaries (dict): Best for mapping unique keys to values, offering fast lookups and mutable data structures for dynamic key-value pairs. 

2. Named Tuples (namedtuple): Ideal for creating lightweight, immutable objects with readable attribute names, serving as a clean and efficient alternative to a dictionary for representing a fixed set of data. 

3. Sets (set): Designed for storing unique, unordered elements, providing very fast membership testing and operations like union and intersection. 

#### In this case, Dictionary would be a better choice than namedtuple or sets

### Step 3: Implement Functions and  Data structure and use them to populate a data structure (Dictionary)


### Step 4: Bulk Loaded - Map data structures from dataframes to dictionaries

In [424]:
# Convert data into a dictionary
records_dict = sales_shipping_data.to_dict(orient="records")
print(records_dict[:3])  # Print the first 3 records to verify



[{'Region': 'Middle East and North Africa', 'Country': 'Libya', 'Item Type': 'Cosmetics', 'Sales Channel': 'Offline', 'Order Priority': 'M', 'Order Date': '10/18/2014', 'Order ID': 686800706, 'Ship Date': '10/31/2014', 'Units Sold': 8446, 'Unit Price': 437.2, 'Unit Cost': 263.33, 'Total Revenue': 3692591.2, 'Total Cost': 2224085.18, 'Total Profit': 1468506.02, 'Shipping Address ID': 85831, 'shipping_address_id': 85831, 'country': 'Libya', 'street': '1069 Main St', 'city': 'Sydney', 'state/province': 'ENG', 'postal_code': 33769.0, 'phone_number': 'Phone Number'}, {'Region': 'North America', 'Country': 'Canada', 'Item Type': 'Vegetables', 'Sales Channel': 'Online', 'Order Priority': 'M', 'Order Date': '11/7/2011', 'Order ID': 185941302, 'Ship Date': '12/8/2011', 'Units Sold': 3018, 'Unit Price': 154.06, 'Unit Cost': 90.93, 'Total Revenue': 464953.08, 'Total Cost': 274426.74, 'Total Profit': 190526.34, 'Shipping Address ID': 72252, 'shipping_address_id': 72252, 'country': 'Canada', 'stree

### Step 5: Quick Profiling - Min/mean/max price, unique city count (set)

In [425]:
numerical_columns = sales_shipping_data.select_dtypes(include=np.float64)
# numerical_columns.head()

# Extract the list of countries
countries = sales_shipping_data['Country']

# Use a set to get the unique countries and count them
unique_countries_count = len(set(countries)) 

print(f"\nTotal number of unique countries: {unique_countries_count}")
print(f"Total number of columns: {len(sales_shipping_data.columns)}")
print(f"Total number of rows: {len(sales_shipping_data)}")

sales_shipping_data.info()

# ======== No need for this since I used df.info() function to print out the details ========
# Calculate statistics for and print the result
# for c in sales_shipping_data.columns:
#     print(f"\n--- Column: '{c}' ---")

#     # Get the count of unique values
#     unique_count = sales_shipping_data[c].nunique()
#     print(f"Unique Values: {unique_count}")

#     # Get the number of missing values
#     missing_count = sales_shipping_data[c].isnull().sum()
#     print(f"Missing Values: {missing_count}")

    
#     if(c in numerical_columns.columns):
#         print(f"Min: ${numerical_columns[c].min():,.2f}")
#         print(f"Max: ${numerical_columns[c].max():,.2f}")
#         print(f"Mean: ${numerical_columns[c].mean():,.2f}")

# sales_shipping_data.head()


Total number of unique countries: 171
Total number of columns: 22
Total number of rows: 500
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 22 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Region               500 non-null    object 
 1   Country              500 non-null    object 
 2   Item Type            500 non-null    object 
 3   Sales Channel        500 non-null    object 
 4   Order Priority       500 non-null    object 
 5   Order Date           500 non-null    object 
 6   Order ID             500 non-null    int64  
 7   Ship Date            500 non-null    object 
 8   Units Sold           500 non-null    int64  
 9   Unit Price           500 non-null    float64
 10  Unit Cost            500 non-null    float64
 11  Total Revenue        500 non-null    float64
 12  Total Cost           500 non-null    float64
 13  Total Profit         500 non-null    float64
 1

#### Step 6: Spot the Grime - Identify at least three dirty data cases

#### Step 7: Cleaning Rules - Execute fixes inside clean(); show “before/after” counts

In [426]:
# Data Cleaning and Normalization

# 1. Remove leading/trailing whitespace from string columns
for col in sales_shipping_data.select_dtypes(include='object').columns:
  sales_shipping_data[col] = sales_shipping_data[col].str.strip()


# 2. Remove the columns 'country' and 'shipping_address_id' since they are dupplicated and they come from both datasets
sales_shipping_data.drop(columns=['country'], inplace=True)
sales_shipping_data.drop(columns=['shipping_address_id'], inplace=True)
    
# 3. Standardize column names (lowercase, replace spaces with underscores)
sales_shipping_data.columns = [col.strip().lower().replace(' ', '_') for col in sales_shipping_data.columns]

# 4. Columns should be numeric whenever possible - replacing online/offline with 1/0
for i in range(len(sales_shipping_data)):
  if sales_shipping_data.loc[i, "sales_channel"].lower() == "online":
    sales_shipping_data.loc[i, "sales_channel"] = 1
  elif sales_shipping_data.loc[i, "sales_channel"].lower() == "offline":
    sales_shipping_data.loc[i, "sales_channel"] = 0
  else:
    sales_shipping_data.loc[i, "sales_channel"] = np.nan

# Check result"
sales_shipping_data.head(len(sales_shipping_data))


,region,country,item_type,sales_channel,order_priority,order_date,order_id,ship_date,units_sold,unit_price,unit_cost,total_revenue,total_cost,total_profit,shipping_address_id,street,city,state/province,postal_code,phone_number
0,Middle East and North Africa,Libya,Cosmetics,0,M,10/18/2014,686800706,10/31/2014,8446,437.20,263.33,3692591.20,2224085.18,1468506.02,85831,1069 Main St,Sydney,ENG,33769.0,Phone Number
1,North America,Canada,Vegetables,1,M,11/7/2011,185941302,12/8/2011,3018,154.06,90.93,464953.08,274426.74,190526.34,72252,8872 Main St,Toronto,NY,98001.0,Phone Number
2,Middle East and North Africa,Libya,Baby Food,0,C,10/31/2016,246222341,12/9/2016,1517,255.28,159.42,387259.76,241840.14,145419.62,92369,2924 Main St,Tokyo,BE,97404.0,Phone Number
3,Asia,Japan,Cereal,0,C,4/10/2010,161442649,5/12/2010,3322,205.70,117.11,683335.40,389039.42,294295.98,56245,1311 Main St,New York,BE,30156.0,Phone Number
4,Sub-Saharan Africa,Chad,Fruits,0,H,8/16/2011,645713555,8/31/2011,9845,9.33,6.92,91853.85,68127.40,23726.45,86737,329 Main St,London,ON,95836.0,Phone Number
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,Middle East and North Africa,Algeria,Meat,0,M,9/2/2011,183022201,10/15/2011,9191,421.89,364.69,3877590.99,3351865.79,525725.20,94611,440 Main St,Tokyo,ON,23406.0,Phone Number
496,Europe,Italy,Personal Care,1,L,3/21/2011,127589738,4/2/2011,5494,81.73,56.67,449024.62,311344.98,137679.64,22835,7522 Main St,Tokyo,ON,29781.0,Phone Number
497,Europe,Russia,Fruits,0,L,1/8/2011,221530139,1/26/2011,4546,9.33,6.92,42414.18,31458.32,10955.86,87376,4233 Main St,Toronto,ENG,67417.0,Phone Number
498,Central America and the Caribbean,Antigua and Barbuda,Office Supplies,0,M,2/22/2015,363329732,2/22/2015,6197,651.21,524.96,4035548.37,3253177.12,782371.25,25966,759 Main St,London,NY,28325.0,Phone Number


In [427]:
# Generate 9 random 5-digit numbers
coupon_codes = np.random.choice(range(10000, 99999), size=9, replace=False)

# Make discount choices, including None for missing values
discount_choices = [5, 10, 15, 20, 25, 30, None]

# Generate exactly 9 discount values
discount_options = np.random.choice(discount_choices, size=9, replace=True)

# Build DataFrame
coupon_df = pd.DataFrame({
    "COUPON CODE": list(coupon_codes),
    "DISCOUNT_PERCENTAGE": list(discount_options)
})

# Save to CSV
coupon_df.to_csv("./data/coupon_codes.csv", index=False)

# Add the new 'Coupon Code' column to the DataFrame
sales_shipping_data['COUPON CODE'] = np.random.choice(coupon_codes, size=len(sales_shipping_data))
print("New 'Coupon Code' column added with random 5-digit numbers!")

# Function to apply discount by going over the data row by row
# When running it this way, it takes 3 seconds to go over 500 rows
# def apply_discount(row):
#     # Load coupon dataset each time (or move outside for efficiency)
#     coupon_csv_path = "./data/coupon_codes.csv"
#     coupon_df = pd.read_csv(coupon_csv_path)

#     condition = coupon_df["COUPON CODE"] == row["COUPON CODE"]

#     selected_row_df = coupon_df.loc[condition]

#     if np.isnan(selected_row_df["DISCOUNT_PERCENTAGE"]).iloc[0]:
#         row["total_revenue_after_disc"] = row["total_revenue"]
#         row["total_profit_after_disc"] = row["total_profit"]
#     else:
#         row["total_revenue_after_disc"] = round(row["total_revenue"] * (1 - selected_row_df["DISCOUNT_PERCENTAGE"].iloc[0] / 100), 2)
#         row["total_profit_after_disc"] = round(row["total_profit"] * (1 - selected_row_df["DISCOUNT_PERCENTAGE"].iloc[0] / 100), 2)

#     return row

# Apply the function row by row
# sales_shipping_data = sales_shipping_data.apply(apply_discount, axis=1)

# When merging the datasets together to appy discounts, it takes maximum 0.3 second to go over 500 rows and apply discounts
def apply_discounts(sales_df):
    coupon_csv_path = "./data/coupon_codes.csv"
    coupons_df = pd.read_csv(coupon_csv_path)

    # Normalize column names for consistency
    coupons_df = coupons_df.rename(columns={"COUPON CODE": "coupon_code"})
    coupons_df = coupons_df.rename(columns={"DISCOUNT_PERCENTAGE": "discount_percentage"})
    
    sales_df = sales_df.rename(columns={"COUPON CODE": "coupon_code"})
    
    # Merge on coupon_code (left join to keep all sales)
    merged = sales_df.merge(coupons_df, on="coupon_code", how="left")

    # Replace missing discounts with 0
    merged["discount_percentage"] = merged["discount_percentage"].fillna(0)

    # Compute discounted values
    merged["total_revenue_after_disc"] = round(
        merged["total_revenue"] * (1 - merged["discount_percentage"] / 100), 2
    )
    merged["total_profit_after_disc"] = round(
        merged["total_profit"] * (1 - merged["discount_percentage"] / 100), 2
    )

    # Removing DISOUNT_PERCENTAGE column
    merged = merged.drop("discount_percentage", axis=1)

    return merged

sales_shipping_data = apply_discounts(sales_shipping_data)

sales_shipping_data.head(len(sales_records_path))

New 'Coupon Code' column added with random 5-digit numbers!


,region,country,item_type,sales_channel,order_priority,order_date,order_id,ship_date,units_sold,unit_price,...,total_profit,shipping_address_id,street,city,state/province,postal_code,phone_number,coupon_code,total_revenue_after_disc,total_profit_after_disc
0,Middle East and North Africa,Libya,Cosmetics,0,M,10/18/2014,686800706,10/31/2014,8446,437.20,...,1468506.02,85831,1069 Main St,Sydney,ENG,33769.0,Phone Number,98507,2584813.84,1027954.21
1,North America,Canada,Vegetables,1,M,11/7/2011,185941302,12/8/2011,3018,154.06,...,190526.34,72252,8872 Main St,Toronto,NY,98001.0,Phone Number,49164,418457.77,171473.71
2,Middle East and North Africa,Libya,Baby Food,0,C,10/31/2016,246222341,12/9/2016,1517,255.28,...,145419.62,92369,2924 Main St,Tokyo,BE,97404.0,Phone Number,98507,271081.83,101793.73
3,Asia,Japan,Cereal,0,C,4/10/2010,161442649,5/12/2010,3322,205.70,...,294295.98,56245,1311 Main St,New York,BE,30156.0,Phone Number,49164,615001.86,264866.38
4,Sub-Saharan Africa,Chad,Fruits,0,H,8/16/2011,645713555,8/31/2011,9845,9.33,...,23726.45,86737,329 Main St,London,ON,95836.0,Phone Number,82743,68890.39,17794.84
5,Europe,Armenia,Cereal,1,H,11/24/2014,683458888,12/28/2014,9528,205.70,...,844085.52,31708,4384 Main St,Berlin,NY,60481.0,Phone Number,95341,1567927.68,675268.42
6,Sub-Saharan Africa,Eritrea,Cereal,1,H,3/4/2015,679414975,4/17/2015,2844,205.70,...,251949.96,16640,3285 Main St,Berlin,ON,23435.0,Phone Number,98507,409507.56,176364.97
7,Europe,Montenegro,Clothes,0,M,5/17/2012,208630645,6/28/2012,7299,109.28,...,536038.56,36315,601 Main St,Sydney,BE,79650.0,Phone Number,72705,558344.30,375226.99
8,Central America and the Caribbean,Jamaica,Vegetables,1,H,1/29/2015,266467225,3/7/2015,2428,154.06,...,153279.64,41310,148 Main St,Sydney,BE,56336.0,Phone Number,95341,299246.14,122623.71
9,Australia and Oceania,Fiji,Vegetables,0,H,12/24/2013,118598544,1/19/2014,4800,154.06,...,303024.00,96451,3666 Main St,New York,NY,46135.0,Phone Number,95341,591590.40,242419.20


#### 9. Feature Engineering - For example: Add days_since_purchase

In [428]:
def add_days_since_purchase(df):
    date_col="order_date"
    # Ensure the column is datetime
    df[date_col] = pd.to_datetime(df[date_col], errors="coerce")

    # Compute days since purchase
    today = pd.Timestamp.today().normalize()  # removes time portion
    df["days_since_purchase"] = (today - df[date_col]).dt.days

    return df

sales_shipping_data = add_days_since_purchase(sales_shipping_data)

sales_shipping_data.head(len(sales_records_path))

,region,country,item_type,sales_channel,order_priority,order_date,order_id,ship_date,units_sold,unit_price,...,shipping_address_id,street,city,state/province,postal_code,phone_number,coupon_code,total_revenue_after_disc,total_profit_after_disc,days_since_purchase
0,Middle East and North Africa,Libya,Cosmetics,0,M,2014-10-18,686800706,10/31/2014,8446,437.20,...,85831,1069 Main St,Sydney,ENG,33769.0,Phone Number,98507,2584813.84,1027954.21,3997
1,North America,Canada,Vegetables,1,M,2011-11-07,185941302,12/8/2011,3018,154.06,...,72252,8872 Main St,Toronto,NY,98001.0,Phone Number,49164,418457.77,171473.71,5073
2,Middle East and North Africa,Libya,Baby Food,0,C,2016-10-31,246222341,12/9/2016,1517,255.28,...,92369,2924 Main St,Tokyo,BE,97404.0,Phone Number,98507,271081.83,101793.73,3253
3,Asia,Japan,Cereal,0,C,2010-04-10,161442649,5/12/2010,3322,205.70,...,56245,1311 Main St,New York,BE,30156.0,Phone Number,49164,615001.86,264866.38,5649
4,Sub-Saharan Africa,Chad,Fruits,0,H,2011-08-16,645713555,8/31/2011,9845,9.33,...,86737,329 Main St,London,ON,95836.0,Phone Number,82743,68890.39,17794.84,5156
5,Europe,Armenia,Cereal,1,H,2014-11-24,683458888,12/28/2014,9528,205.70,...,31708,4384 Main St,Berlin,NY,60481.0,Phone Number,95341,1567927.68,675268.42,3960
6,Sub-Saharan Africa,Eritrea,Cereal,1,H,2015-03-04,679414975,4/17/2015,2844,205.70,...,16640,3285 Main St,Berlin,ON,23435.0,Phone Number,98507,409507.56,176364.97,3860
7,Europe,Montenegro,Clothes,0,M,2012-05-17,208630645,6/28/2012,7299,109.28,...,36315,601 Main St,Sydney,BE,79650.0,Phone Number,72705,558344.30,375226.99,4881
8,Central America and the Caribbean,Jamaica,Vegetables,1,H,2015-01-29,266467225,3/7/2015,2428,154.06,...,41310,148 Main St,Sydney,BE,56336.0,Phone Number,95341,299246.14,122623.71,3894
9,Australia and Oceania,Fiji,Vegetables,0,H,2013-12-24,118598544,1/19/2014,4800,154.06,...,96451,3666 Main St,New York,NY,46135.0,Phone Number,95341,591590.40,242419.20,4295


#### 10. Mini-Aggregation - For example: Revenue per shipping_city (dict or pandas.groupby)

In [429]:
"""
Cell generated by Data Wrangler.
"""
def clean_data(df):
    # Performed 1 aggregation grouped on column: 'city'
    df = df.groupby(['city']).agg(total_revenue_sum=('total_revenue', 'sum')).reset_index()
    return df

# Loaded variable 'df' from kernel state
df = sales_shipping_data.head(len(sales_records_path))

df_clean = clean_data(df.copy())
df_clean.head()

,city,total_revenue_sum
0,Berlin,15941828.83
1,London,9125712.05
2,New York,13896148.76
3,Sydney,5715761.86
4,Tokyo,9492139.30


#### 11. Serialization Checkpoint - Save cleaned data to JSON

In [430]:
# Convert to JSON (records format is usually the most useful)
json_str = sales_shipping_data.to_json(orient="records", indent=2)

print(json_str)

[
  {
    "region":"Middle East and North Africa",
    "country":"Libya",
    "item_type":"Cosmetics",
    "sales_channel":0,
    "order_priority":"M",
    "order_date":1413590400000,
    "order_id":686800706,
    "ship_date":"10\/31\/2014",
    "units_sold":8446,
    "unit_price":437.2,
    "unit_cost":263.33,
    "total_revenue":3692591.2000000002,
    "total_cost":2224085.1800000002,
    "total_profit":1468506.02,
    "shipping_address_id":85831,
    "street":"1069 Main St",
    "city":"Sydney",
    "state\/province":"ENG",
    "postal_code":33769.0,
    "phone_number":"Phone Number",
    "coupon_code":98507,
    "total_revenue_after_disc":2584813.8399999999,
    "total_profit_after_disc":1027954.21,
    "days_since_purchase":3997
  },
  {
    "region":"North America",
    "country":"Canada",
    "item_type":"Vegetables",
    "sales_channel":1,
    "order_priority":"M",
    "order_date":1320624000000,
    "order_id":185941302,
    "ship_date":"12\/8\/2011",
    "units_sold":3018,
  

#### 12. Soft Interview Reflection - Markdown: < 120 words explaining how Functions have helped 